# Base Student Output Generation - Google Colab
Run on T4 GPU for faster generation. Continues from where local script left off.

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth supabase python-dotenv -q
!pip install --upgrade transformers -q

In [ ]:
# Cell 2: Set your Supabase credentials
# Get these from your .env file or Supabase dashboard

SUPABASE_URL = "YOUR_SUPABASE_URL"  # e.g., https://xxxxx.supabase.co
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_KEY"  # Use service role key for updates

In [ ]:
# Cell 3: Load model (4-bit quantized for speed)
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True

from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print("Loading model...")
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded on {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Cell 4: Connect to Supabase and check progress
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Count remaining records
result = supabase.table("modelcomp_50k").select("id", count="exact").is_("student_output", "null").execute()
remaining = result.count
print(f"Records still needing student_output: {remaining}")

In [ ]:
# Cell 5: Generation function
import time

MAX_NEW_TOKENS = 256

def generate_output(instruction: str, context: str = "") -> tuple:
    """Generate output and return (text, latency)"""
    start_time = time.time()
    
    # Build prompt
    if context:
        user_content = f"Context: {context}\n\nInstruction: {instruction}"
    else:
        user_content = instruction
    
    messages = [
        {"role": "user", "content": user_content}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the new tokens
    new_tokens = outputs[0][inputs.shape[1]:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    latency = time.time() - start_time
    return output_text.strip(), latency

In [ ]:
# Cell 6: Test generation (run this first to verify it works)
test_output, test_latency = generate_output("What is machine learning?", "")
print(f"Test output ({test_latency:.2f}s):\n{test_output[:200]}...")

In [ ]:
# Cell 7: Main generation loop - processes all remaining records
from tqdm import tqdm

BATCH_SIZE = 100  # Fetch 100 at a time from Supabase
total_processed = 0
errors = []

print("Starting generation...")
print("Progress saves after EACH record - safe to stop anytime!")
print("-" * 50)

while True:
    # Fetch batch of records without student_output
    result = supabase.table("modelcomp_50k") \
        .select("id, input, context") \
        .is_("student_output", "null") \
        .limit(BATCH_SIZE) \
        .execute()
    
    records = result.data
    if not records:
        print("\n✅ All records processed!")
        break
    
    print(f"\nProcessing batch of {len(records)} records...")
    
    for record in tqdm(records, desc="Generating"):
        try:
            record_id = record["id"]
            instruction = record["input"] or ""
            context = record["context"] or ""
            
            # Generate
            output, latency = generate_output(instruction, context)
            
            # Save immediately to Supabase
            supabase.table("modelcomp_50k").update({
                "student_output": output,
                "generation_latency": round(latency, 3)
            }).eq("id", record_id).execute()
            
            total_processed += 1
            
        except Exception as e:
            errors.append({"id": record_id, "error": str(e)})
            print(f"\nError on {record_id}: {e}")
    
    # Progress update
    remaining_result = supabase.table("modelcomp_50k").select("id", count="exact").is_("student_output", "null").execute()
    print(f"Processed: {total_processed} | Remaining: {remaining_result.count} | Errors: {len(errors)}")

print(f"\n{'='*50}")
print(f"Total processed this session: {total_processed}")
print(f"Errors: {len(errors)}")

In [ ]:
# Cell 8: Check final progress
done = supabase.table("modelcomp_50k").select("id", count="exact").not_.is_("student_output", "null").execute()
remaining = supabase.table("modelcomp_50k").select("id", count="exact").is_("student_output", "null").execute()

print(f"✅ Done: {done.count}")
print(f"⏳ Remaining: {remaining.count}")
print(f"Progress: {done.count / (done.count + remaining.count) * 100:.1f}%")